In [1]:
from ase import Atoms
from pathlib import Path
import numpy as np
from typing import Union

In [2]:
def get_ase_atoms_from_fdmnes(file_path: Union[str, Path]) -> Atoms:
    '''
    Читает структуру НЧ (молекулы) из файла FDMNES и возвращает объект ASE Atoms.
    '''
    with open(file_path, 'r') as f:
        lines = f.readlines()
        
    lines_array = np.array(lines)
    molecule_marker_idx = np.where(lines_array == 'Molecule\n')[0][0]
    
    # После маркера идут 6 параметров ячейки: a, b, c, alpha, beta, gamma
    cell_parameters = np.array(lines_array[molecule_marker_idx + 1].split(), dtype=float)
    
    # Вычисляем декартовы базисные векторы
    lattice_vector_a, lattice_vector_b, lattice_vector_c = convert_cell_params_to_vectors(cell_parameters)
    
    lattice_vector_a = np.round(lattice_vector_a, 10)
    lattice_vector_b = np.round(lattice_vector_b, 10)
    lattice_vector_c = np.round(lattice_vector_c, 10)
    
    # Данные об атомах [атомный_номер, x_frac, y_frac, z_frac]
    atom_data_lines = lines[molecule_marker_idx + 2:]
    atom_data_flat = np.fromstring("".join(atom_data_lines), dtype=float, sep="\n")
    
    num_atoms = atom_data_flat.shape[0] // 4
    atom_matrix = atom_data_flat.reshape(num_atoms, 4)
    
    atomic_numbers = np.round(atom_matrix[:, 0]).astype(int)
    fractional_positions = atom_matrix[:, 1:4]
    
    # Переводим кристаллографические координаты в декартовы
    cartesian_positions = (fractional_positions[:, 0, None] * lattice_vector_a +
                           fractional_positions[:, 1, None] * lattice_vector_b +
                           fractional_positions[:, 2, None] * lattice_vector_c
                          )

    # Создаем объект Atoms 
    atoms = Atoms(f"H{len(atomic_numbers)}")
    atoms.set_atomic_numbers(atomic_numbers)
    atoms.set_positions(cartesian_positions)
    atoms.pbc = False
    atoms.set_cell((lattice_vector_a, lattice_vector_b, lattice_vector_c))
    
    return atoms

    
def convert_cell_params_to_vectors(cell_params: np.ndarray):
    '''
    Конвертирует параметры ячейки [a, b, c, alpha, beta, gamma] 
    в декартовы базисные векторы ячейки.
    '''
    assert cell_params.shape == (6,), f"cell_params.shape = {cell_params.shape} != 6"
    
    # Извлекаем длины сторон и углы
    a_len, b_len, c_len = cell_params[0:3]
    angles_rad = cell_params[3:6] / 180 * np.pi
    alpha, beta, gamma = angles_rad[0], angles_rad[1], angles_rad[2]
    
    # Строим вектор b вдоль оси Y
    vec_b = np.array([0.0, b_len, 0.0])
    
    # Строим вектор c в плоскости YZ
    vec_c = np.array([0.0, c_len * np.cos(alpha), c_len * np.sin(alpha)])
    
    # Находим компоненты вектора a из скалярных произведений
    vec_a = np.zeros(3)
    vec_a[1] = a_len * b_len * np.cos(gamma) / vec_b[1]
    vec_a[2] = (a_len * c_len * np.cos(beta) - vec_a[1] * vec_c[1]) / vec_c[2]
    vec_a[0] = np.sqrt(a_len**2 - vec_a[1]**2 - vec_a[2]**2)
    
    return vec_a, vec_b, vec_c